# 06 — Structure and controls

Schrenk-like lift loading, no inertia relief, and a declared safety factor screen
the carbon tubes. These checks omit joint/adhesive failure, local print strength,
buckling and fatigue. Servo torque uses the documented RC empirical rule with
an explicit margin; bench linkage loads remain required.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=140)
    plt.show()


In [ ]:
from uavdesign import structures as struct
from uavdesign.fab.geometry import WingGeometry, spar_layout
spar = d.design["structure"]["spars"]
geometry = WingGeometry.from_airframe(D.airframe_geometry(d))
layout = spar_layout(geometry,spar,pl["pod_width_mm"]/2)
stations = np.linspace(0,d.wing.half_span_mm,studies["plot_points"])
moments = np.array([struct.root_bending_moment_n_mm(d.selected.auw_kg*G,d.wing.half_span_mm,
                  req["load_factor_limit"], y,pl["taper"])*req["safety_factor"] for y in stations])
rows = []
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(stations,moments,label="Factored bending moment")
for name, station in (("joiner",0),("sleeve",pl["pod_width_mm"]/2),
                      ("outer_spar",spar["outer_spar"]["y_start_fraction"]*d.wing.half_span_mm)):
    tube = struct.Tube(spar[name]["od_mm"],spar[name]["id_mm"],spar["density_g_cm3"],spar["allowable_mpa"],spar["modulus_gpa"])
    check = struct.check_spar(tube,d.selected.auw_kg*G,d.wing.half_span_mm,req["load_factor_limit"],req["safety_factor"],station,pl["taper"])
    rows.append({"tube":name,**check.__dict__})
    capacity=tube.allowable_mpa*tube.second_moment_mm4/(tube.od_mm/2)
    axes[0].axhline(capacity,ls="--",label=f"{name} section capacity")
axes[0].set(xlabel="Span station [mm]",ylabel="Moment [N mm]");axes[0].legend()
ys = np.linspace(0,d.wing.half_span_mm,studies["plot_points"])
axes[1].plot(ys,[geometry.le_x(y) for y in ys],color="black")
axes[1].plot(ys,[geometry.le_x(y)+geometry.chord(y) for y in ys],color="black")
for name in ("joiner","sleeve","outer","rear_pin"):
    path=getattr(layout,name)
    axes[1].plot([path.p0[1],path.p1[1]],[path.p0[0],path.p1[0]],label=name)
axes[1].invert_yaxis();axes[1].set(xlabel="Span [mm]",ylabel="x aft [mm]");axes[1].legend()
save(fig,"06_spars.png")
display(pd.DataFrame(rows));display(layout.checks)
screen=studies["servo_screen"]
start,end=pl["elevon_span_fraction"]
surface_chord=float(d.wing.chord_at((start+end)/2*d.wing.half_span_mm))*pl["elevon_chord_fraction"]
torque=struct.servo_torque_kg_cm(surface_chord,(end-start)*d.wing.half_span_mm,
        d.selected.cruise_speed_m_s,screen["surface_deflection_deg"],screen["servo_deflection_deg"])
display(pd.DataFrame([{"required_with_margin_kg_cm":torque*screen["torque_margin"],
                      "rated_kg_cm":screen["rated_torque_kg_cm"],
                      "pass":torque*screen["torque_margin"]<=screen["rated_torque_kg_cm"],
                      "status":screen["status"]}]))
